In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [1]:
suppressPackageStartupMessages({
    library("ibd")
    library("crossdes")
    library("nlme")
    library("lme4")
})

## ___Power analysis___
-----------------

1. __Sympatric host ecotype – AM fungal community pairs will demonstrate superior adaptive fitness (measured by total plant biomass, plant height, PRLC & total length of hyphae in gram soil) compared to their allopatric counterparts__

2. __The extent of outsourcing reflected in the fine absorptive root traits (RD, SRL & RCT) will be the highest for each plant ecotype when paired with sympatric AM fungal community compared to pairings with foreign AM fungal communities.__

-----------------------

3. __Amongst the sympatric AM fungal-host ecotype pairs, the magnitude of this influence will be more pronounced where the hosts needed the AM fungi more (e.g. low phosphorus soils).__

4. __The fine absorptive root traits (collaboration gradient) will also be influenced by the AM fungal community composition in the soil samples, but this influence will be less pronounced compared to the provenance specific coadaptations.__

In [2]:
# have a look at the root trait data from Vin
themeda <- read.csv("../data/chapter3/vin_themeda_root_traits.csv", stringsAsFactors = TRUE)
themeda

ID,Accession,Rep,Weight..mg.,Weight..g.,Length.cm.,Length.m.,SurfArea.cm2.,AvgDiam.mm.,LenPerVol.cm.m3.,RootVolume.cm3.,SRL.m.g.,SRA.cm2.g.,RTD.gcm3.,SRL,SRA,RTD,Diameter
<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Dalby-E,Dalby_QLD,E,368.4,0.3684,600.2167,6.002167,104.1133,0.552100,600.2167,1.437,16.29253,282.6094,0.25636743,16.29253,282.6094,0.25636743,0.552100
Dalby-F,Dalby_QLD,F,228.3,0.2283,812.4593,8.124593,126.7304,0.496500,812.4593,1.573,35.58735,555.1047,0.14513668,35.58735,555.1047,0.14513668,0.496500
Dalby-G,Dalby_QLD,G,279.9,0.2799,735.3292,7.353292,88.1581,0.381600,735.3292,0.841,26.27114,314.9628,0.33281807,26.27114,314.9628,0.33281807,0.381600
Dalby-H,Dalby_QLD,H,380.4,0.3804,662.1124,6.621124,134.1011,0.644700,662.1124,2.161,17.40569,352.5266,0.17602962,17.40569,352.5266,0.17602962,0.644700
55830-E,Mt Fox N Park_QLD,E,292.7,0.2927,1027.2178,10.272178,119.3707,0.369900,1027.2178,1.104,35.09456,407.8261,0.26512681,35.09456,407.8261,0.26512681,0.369900
55830-F,Mt Fox N Park_QLD,F,304.5,0.3045,721.0387,7.210387,113.3194,0.500300,721.0387,1.417,23.67943,372.1491,0.21489061,23.67943,372.1491,0.21489061,0.500300
55830-H,Mt Fox N Park_QLD,H,296.1,0.2961,1022.2098,10.222098,112.9964,0.351900,1022.2098,0.994,34.52245,381.6157,0.29788732,34.52245,381.6157,0.29788732,0.351900
55830-I,Mt Fox N Park_QLD,I,333.8,0.3338,580.7923,5.807923,112.0177,0.613900,580.7923,1.719,17.39941,335.5833,0.19418266,17.39941,335.5833,0.19418266,0.613900
Panawonica-E,Panawonica_WA,E,223.7,0.2237,475.1663,4.751663,47.3264,0.317000,475.1663,0.375,21.24123,211.5619,0.59653333,21.24123,211.5619,0.59653333,0.317000


In [10]:
# examine the site means
themeda |> split(~Accession) |> lapply(FUN = function (df){ c(mean(df$SRL), mean(df$Diameter))}) |> as.data.frame(row.names = c("SRL", "RD"))

,Dalby_QLD,Mt.Fox.N.Park_QLD,Panawonica_WA,Rainbow.Valley_NT,Sydney,Virginia.Gardens_SA
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
SRL,23.889177,27.67396,18.01079,28.71809,37.4110131,31.0229188
RD,0.518725,0.45900,0.62180,0.50114,0.4763333,0.5734438


In [11]:
# focusing on SRL, let's pick the two sites that show the largest variations
# Panawonica_WA - the smallest mean SRL
# Sydney - the largest mean SRL

In [ ]:
# now, if we were to expect this level of variation between the sympatric and allopatric groups,......
# we can generate random response (SRL) values for the sympatric and allopatric groups, levereging the distribition (assuming normality????) of these two sites

In [42]:
# CONSTANTS

PAIRS6X6 <- expand.grid(1:6, 1:6)
colnames(PAIRS6X6) <- c("seed", "soil")
IS_SYMPATRIC = (PAIRS6X6$seed == PAIRS6X6$soil)

# parameters to specify a realistic random dist for SRL
MEAN_SRL <- mean(themeda$SRL)
STD_SRL <- sd(themeda$SRL)

NREPLICATES <- seq(from = 3, to = 12) # a realistic range of replicates we can afford to have in the experiment
EFFECT_SIZES <- seq(from = 0.1, to = 0.9, length.out = 30) # range of effect sizes to test
NITERATIONS <- 1000 # number of iterations to average the power over

In [37]:
average_power <- matrix(ncol = length(NREPLICATES), nrow = length(EFFECT_SIZES))

In [207]:
for (nreps in NREPLICATES) {
        df <- PAIRS6X6[rep(1:36, nreps), ] # expand the PAIRS6X6 dataframe such that each row gets repcilated nreps times
        bmask_sympatric <- rep(IS_SYMPATRIC, nreps) # expand the boolean mask for sympatric records by the same factor
    for (efsize in EFFECT_SIZES) {
        
    for (i in 1:NITERATIONS) { # NITERATIONS repeated random draws
        
    }
        
    }
}

# gsep - geographical separation
for (i in 1:nrow(PAIRS6X6)) {
    # if the pair is sympatric, update the SRL using a normal dist representing Panawonica_WA's SRL values
    if (substr(PAIRS6X6[i, "seed"], start = 3, stop = 3) == substr(PAIRS6X6[i, "soil"], start = 4, stop = 4)) { 
        PAIRS6X6[i, "SRL"] <- PAIRS6X6[i, "pairwise_noise"] * rnorm(mean = 18.010786, sd = 4.721772, n = 1)
        # ignore the pairwie noise for now
        # multuply the rnorm() output by 0.9
        PAIRS6X6[i, "gsep"] <- 'S' # Sympatric
    } else { # if it is allopatric, update the SRL using a normal dist representing Sydney's SRL values
        PAIRS6X6[i, "SRL"] <- PAIRS6X6[i, "pairwise_noise"] * rnorm(mean = 37.411013, sd = 9.127129, n = 1)
        PAIRS6X6[i, "gsep"] <- 'A' # Allopatric
        # else multiply thr rnorm() output by 1.0
    }                                     
}

In [208]:
PAIRS6X6 |> head()

,seed,soil,pairwise_noise,SRL,gsep
,<fct>,<fct>,<dbl>,<dbl>,<chr>
1,TT1,AMF1,1.303793,13.91872,S
2,TT2,AMF1,1.425303,41.28782,A
3,TT3,AMF1,1.408745,77.85428,A
4,TT4,AMF1,1.332403,58.99182,A
5,TT5,AMF1,1.357731,43.02939,A
6,TT6,AMF1,1.458698,63.01665,A


In [214]:
# allopatric vs sympatric becomes our fixed effect => ~type
# seed and soil origins become our (non nested) random effects
# https://stats.stackexchange.com/questions/674034/statistical-test-for-significance-of-mean-differences-between-two-groups
mixmod <- lme4::lmer("SRL~gsep+(1|seed)+(1|soil)", data = PAIRS6X6, REML = TRUE)

boundary (singular) fit: see help('isSingular')



In [215]:
mixmod

Linear mixed model fit by REML ['lmerMod']
Formula: SRL ~ gsep + (1 | seed) + (1 | soil)
   Data: dummy_results
REML criterion at convergence: 5620.773
Random effects:
 Groups   Name        Std.Dev. 
 seed     (Intercept) 0.000e+00
 soil     (Intercept) 9.092e-07
 Residual             1.203e+01
Number of obs: 720, groups:  seed, 6; soil, 6
Fixed Effects:
(Intercept)        gsepS  
      52.48       -27.64  
optimizer (nloptwrap) convergence code: 0 (OK) ; 0 optimizer warnings; 1 lme4 warnings 

In [216]:
summary(mixmod)

Linear mixed model fit by REML ['lmerMod']
Formula: SRL ~ gsep + (1 | seed) + (1 | soil)
   Data: dummy_results

REML criterion at convergence: 5620.8

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5880 -0.6485 -0.0038  0.5900  3.4103 

Random effects:
 Groups   Name        Variance  Std.Dev. 
 seed     (Intercept) 0.000e+00 0.000e+00
 soil     (Intercept) 8.266e-13 9.092e-07
 Residual             1.447e+02 1.203e+01
Number of obs: 720, groups:  seed, 6; soil, 6

Fixed effects:
            Estimate Std. Error t value
(Intercept)  52.4777     0.4912  106.85
gsepS       -27.6368     1.2031  -22.97

Correlation of Fixed Effects:
      (Intr)
gsepS -0.408
optimizer (nloptwrap) convergence code: 0 (OK)
boundary (singular) fit: see help('isSingular')


## ___Balanced Incomplete Block Design (BIBD)___
--------------------------------

In [3]:
# reference
# https://people.math.ethz.ch/~meier/teaching/anova/incomplete-block-designs.html

In [7]:
# let's say that we need 36 reps for each allopatric and sympatric groups
# the sympatric 36 can be easily divided between the 6 pairs (6 reps per each pair)
# how do we divide the other 36 between the 30 potential pairs????

In [ ]:
# treat the remaining 30 pairs as 30 individual units, numbered 1 to 30
# 

In [ ]:
# treat the seeds as blocks => 6 blocks
# each block will be tested against 6 soil treatments (in a complete block design)
# say that in an IBD, each seed only gets three soil treatments

In [21]:
# - v: number of treatments
# - b: number of blocks
# - r: number of replicates (across all blocks)
# - k: number of experimental units per block
# - lambda: lambda

ibd::bibd(v=5, b=5, k=3, r=3, lambda=2)

[1] "parameters do not satisfy necessary conditions"

In [25]:
ibd::ibd(v=6, b=6, k=4)

$v
[1] 6

$b
[1] 6

$k
[1] 4

$NNP
     [,1] [,2] [,3] [,4] [,5] [,6]
[1,]    4    2    2    2    3    3
[2,]    2    4    3    2    2    3
[3,]    2    3    4    3    2    2
[4,]    2    2    3    4    3    2
[5,]    3    2    2    3    4    2
[6,]    3    3    2    2    2    4

$N
     [,1] [,2] [,3] [,4] [,5] [,6]
[1,]    1    1    1    0    0    1
[2,]    0    0    1    1    1    1
[3,]    0    1    0    1    1    1
[4,]    1    1    0    1    1    0
[5,]    1    1    1    0    1    0
[6,]    1    0    1    1    0    1

$design
        [,1] [,2] [,3] [,4]
Block-1    1    4    5    6
Block-2    1    3    4    5
Block-3    1    2    5    6
Block-4    2    3    4    6
Block-5    2    3    4    5
Block-6    1    2    3    6

$conc.mat
     [,1] [,2] [,3] [,4] [,5] [,6]
[1,]    4    2    2    2    3    3
[2,]    2    4    3    2    2    3
[3,]    2    3    4    3    2    2
[4,]    2    2    3    4    3    2
[5,]    3    2    2    3    4    2
[6,]    3    3    2    2    2    4

$A.Effici